In [4]:
import pandas as pd
import numpy as np

print("==================================================")
print("SPAIN TOP 50 SONGS: ANALYTICAL METHODOLOGY")
print("==================================================")

# ----------------------------------------------------
# STEP 1: Data Validation & Normalization
# ----------------------------------------------------
print("\n[Step 1] Loading and Normalizing Dataset...")
df = pd.read_csv('Atlantic_Spain.csv')

# Convert date column to datetime
df['date_dt'] = pd.to_datetime(df['date'], format='%d-%m-%Y')

# Verify daily record counts and handle duplicates if any
daily_counts = df.groupby('date_dt')['position'].count()
if not all(daily_counts == 50):
    print("-> Note: Duplicate snapshots detected on certain dates. Cleaning duplicates...")
    df = df.drop_duplicates(subset=['date_dt', 'position'], keep='first').copy()

# Normalize song and artist strings
df['song_norm'] = df['song'].str.strip().str.title()
df['artist_norm'] = df['artist'].str.strip().str.title()
df['song_id'] = df['song_norm'] + " - " + df['artist_norm']

print(f"-> Total Cleaned Records: {len(df)}")
print(f"-> Total Unique Snapshot Days: {df['date_dt'].nunique()}")
print(f"-> Total Unique Songs: {df['song_id'].nunique()}")

# ----------------------------------------------------
# STEP 2: Song Lifecycle Construction
# ----------------------------------------------------
print("\n[Step 2] Constructing Song Lifecycles...")

lifecycle_df = df.groupby('song_id').agg(
    entry_date=('date_dt', 'min'),
    exit_date=('date_dt', 'max'),
    days_present=('date_dt', 'nunique'),
    span_days=('date_dt', lambda x: (x.max() - x.min()).days + 1),
    peak_position=('position', 'min'),
    avg_popularity=('popularity', 'mean'),
    album_type=('album_type', 'first'),
    total_tracks=('total_tracks', 'first'),
    is_explicit=('is_explicit', 'first'),
    duration_ms=('duration_ms', 'first')
).reset_index()

# Calculate time to peak (days from entry to first reaching peak position)
def get_time_to_peak(group):
    peak_val = group['position'].min()
    peak_date = group[group['position'] == peak_val]['date_dt'].min()
    entry_date = group['date_dt'].min()
    return (peak_date - entry_date).days

time_to_peak_series = df.groupby('song_id').apply(get_time_to_peak)
lifecycle_df['days_to_peak'] = lifecycle_df['song_id'].map(time_to_peak_series)

print("-> Song Lifecycle Summary Statistics:")
print(lifecycle_df[['days_present', 'span_days', 'days_to_peak', 'peak_position']].describe())

# ----------------------------------------------------
# STEP 3: Lifecycle Stage Classification (Daily Snapshot Level)
# ----------------------------------------------------
print("\n[Step 3] Classifying Lifecycle Stages...")

def classify_lifecycle_stage(row, song_stats):
    # Retrieve song-level metrics for context
    days_active = row['days_active_so_far']
    pos = row['position']
    peak = row['peak_position']
    
    if days_active <= 7:
        return 'New Entry (≤7d)'
    elif pos <= 10 and pos <= peak + 2:
        return 'Peak Phase (Top 10)'
    elif pos < row.get('prev_pos', pos):
        return 'Growth Phase'
    elif 11 <= pos <= 30:
        return 'Mature Phase'
    else:
        return 'Decline Phase'

# Simplified overall song lifecycle categorization based on peak and retention
def categorize_overall_lifecycle(row):
    if row['days_present'] <= 7:
        return 'Short-Term Entry'
    elif row['peak_position'] <= 10:
        return 'Top 10 Hit'
    elif row['avg_popularity'] > 75:
        return 'High-Popularity Stable'
    else:
        return 'Standard Rotation'

lifecycle_df['lifecycle_category'] = lifecycle_df.apply(categorize_overall_lifecycle, axis=1)
print(lifecycle_df['lifecycle_category'].value_counts())

# ----------------------------------------------------
# STEP 4: Playlist Rotation & Churn Analysis
# ----------------------------------------------------
print("\n[Step 4] Calculating Playlist Rotation & Churn...")

dates = sorted(df['date_dt'].unique())
churn_records = []

for i in range(1, len(dates)):
    prev_songs = set(df[df['date_dt'] == dates[i-1]]['song_id'])
    curr_songs = set(df[df['date_dt'] == dates[i]]['song_id'])
    exits = len(prev_songs - curr_songs)
    entries = len(curr_songs - prev_songs)
    churn_records.append({
        'date': dates[i],
        'exits': exits,
        'entries': entries,
        'churn_rate': exits / 50.0
    })

churn_df = pd.DataFrame(churn_records)
print(f"-> Average Daily Exits/Entries: {churn_df['exits'].mean():.2f} songs/day")
print(f"-> Average Daily Churn Rate: {churn_df['churn_rate'].mean() * 100:.2f}%")

# ----------------------------------------------------
# STEP 5: Content Attribute vs. Lifecycle Analysis
# ----------------------------------------------------
print("\n[Step 5] Analyzing Content Attributes (Explicit vs. Clean & Single vs. Album)...")

print("\n--- Explicit vs. Clean Longevity ---")
explicit_summary = lifecycle_df.groupby('is_explicit').agg(
    song_count=('song_id', 'count'),
    mean_days_present=('days_present', 'mean'),
    mean_span_days=('span_days', 'mean'),
    mean_peak_pos=('peak_position', 'mean'),
    mean_popularity=('avg_popularity', 'mean')
)
print(explicit_summary)

print("\n--- Single vs. Album Track Longevity ---")
album_summary = lifecycle_df.groupby('album_type').agg(
    song_count=('song_id', 'count'),
    mean_days_present=('days_present', 'mean'),
    mean_span_days=('span_days', 'mean'),
    mean_peak_pos=('peak_position', 'mean'),
    mean_popularity=('avg_popularity', 'mean')
)
print(album_summary)

print("\n==================================================")
print("ANALYTICAL METHODOLOGY EXECUTION COMPLETED SUCCESSFULLY")
print("==================================================")

SPAIN TOP 50 SONGS: ANALYTICAL METHODOLOGY

[Step 1] Loading and Normalizing Dataset...
-> Note: Duplicate snapshots detected on certain dates. Cleaning duplicates...
-> Total Cleaned Records: 27750
-> Total Unique Snapshot Days: 555
-> Total Unique Songs: 575

[Step 2] Constructing Song Lifecycles...


C:\Users\nidhi\AppData\Local\Temp\ipykernel_15304\646191188.py:57: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  time_to_peak_series = df.groupby('song_id').apply(get_time_to_peak)


-> Song Lifecycle Summary Statistics:
       days_present   span_days  days_to_peak  peak_position
count    575.000000  575.000000    575.000000     575.000000
mean      48.260870   57.443478      9.483478      21.793043
std       73.750886   89.967337     30.019436      14.976775
min        1.000000    1.000000      0.000000       1.000000
25%        4.000000    4.000000      0.000000       9.000000
50%       13.000000   15.000000      0.000000      20.000000
75%       69.000000   82.500000      7.500000      35.000000
max      538.000000  551.000000    426.000000      50.000000

[Step 3] Classifying Lifecycle Stages...
lifecycle_category
Short-Term Entry          237
Top 10 Hit                148
High-Popularity Stable    113
Standard Rotation          77
Name: count, dtype: int64

[Step 4] Calculating Playlist Rotation & Churn...
-> Average Daily Exits/Entries: 1.72 songs/day
-> Average Daily Churn Rate: 3.44%

[Step 5] Analyzing Content Attributes (Explicit vs. Clean & Single vs. A

In [6]:
"""
Spain Top 50 Songs: Content Maturity, Release Lifecycle & Playlist Rotation Analysis
Author: Atlantic Recording Corporation Analytics Team
Dataset: Atlantic_Spain.csv
"""

import pandas as pd
import numpy as np

def run_analysis():
    print("================================================================")
    print("STARTING SPAIN TOP 50 LIFECYCLE & CONTENT MATURITY ANALYSIS")
    print("================================================================")

    # 1. Load Dataset
    df = pd.read_csv('Atlantic_Spain.csv')
    df['date_dt'] = pd.to_datetime(df['date'], format='%d-%m-%Y')

    # 2. Data Validation & Normalization
    # Remove duplicate snapshots if any exist on the same date and position
    initial_rows = len(df)
    df = df.drop_duplicates(subset=['date_dt', 'position'], keep='first').copy()
    print(f"-> Loaded {initial_rows} rows. Cleaned duplicates: {initial_rows - len(df)} rows removed.")
    
    # Normalize song and artist names
    df['song_norm'] = df['song'].str.strip().str.title()
    df['artist_norm'] = df['artist'].str.strip().str.title()
    df['song_id'] = df['song_norm'] + " - " + df['artist_norm']

    # 3. Song Lifecycle Construction
    song_summary = df.groupby('song_id').agg(
        entry_date=('date_dt', 'min'),
        exit_date=('date_dt', 'max'),
        days_present=('date_dt', 'nunique'),
        span_days=('date_dt', lambda x: (x.max() - x.min()).days + 1),
        peak_pos=('position', 'min'),
        avg_pos=('position', 'mean'),
        avg_popularity=('popularity', 'mean'),
        album_type=('album_type', 'first'),
        total_tracks=('total_tracks', 'first'),
        is_explicit=('is_explicit', 'first'),
        duration_ms=('duration_ms', 'first')
    ).reset_index()

    # Calculate Time-to-Peak
    def calc_time_to_peak(group):
        peak_val = group['position'].min()
        peak_date = group[group['position'] == peak_val]['date_dt'].min()
        entry_date = group['date_dt'].min()
        return (peak_date - entry_date).days

    time_to_peak = df.groupby('song_id').apply(calc_time_to_peak)
    song_summary['days_to_peak'] = song_summary['song_id'].map(time_to_peak)

    # Calculate Retention Stability Index
    song_summary['retention_stability_index'] = song_summary['days_present'] / song_summary['span_days']

    # 4. Playlist Churn Calculation
    dates = sorted(df['date_dt'].unique())
    daily_churn = []
    for i in range(1, len(dates)):
        prev_songs = set(df[df['date_dt'] == dates[i-1]]['song_id'])
        curr_songs = set(df[df['date_dt'] == dates[i]]['song_id'])
        exits = len(prev_songs - curr_songs)
        daily_churn.append(exits / 50.0)
    avg_churn_rate = np.mean(daily_churn)

    # 5. Key Performance Indicators (KPIs) Calculation
    avg_days_on_playlist = song_summary['days_present'].mean()
    avg_time_to_peak = song_summary['days_to_peak'].mean()
    avg_retention_stability = song_summary['retention_stability_index'].mean()

    explicit_days = song_summary[song_summary['is_explicit'] == True]['days_present'].mean()
    clean_days = song_summary[song_summary['is_explicit'] == False]['days_present'].mean()
    explicit_score = explicit_days / clean_days

    single_days = song_summary[song_summary['album_type'] == 'single']['days_present'].mean()
    album_days = song_summary[song_summary['album_type'] == 'album']['days_present'].mean()
    single_album_ratio = single_days / album_days

    # Print Executive KPI Summary
    print("\n----------------------------------------------------------------")
    print("KEY PERFORMANCE INDICATORS (KPIS)")
    print("----------------------------------------------------------------")
    print(f"• Average Days on Playlist:       {avg_days_on_playlist:.2f} Days")
    print(f"• Entry-to-Peak Time:             {avg_time_to_peak:.2f} Days")
    print(f"• Playlist Churn Rate:            {avg_churn_rate * 100:.2f}% per day")
    print(f"• Retention Stability Index:      {avg_retention_stability:.3f} ({avg_retention_stability * 100:.1f}%)")
    print(f"• Explicit Content Lifecycle Score: {explicit_score:.3f} ({explicit_days:.1f}d explicit vs {clean_days:.1f}d clean)")
    print(f"• Single vs Album Longevity Ratio:  {single_album_ratio:.2f}x ({single_days:.1f}d single vs {album_days:.1f}d album)")
    print("----------------------------------------------------------------")

    # 6. Attribute Comparisons
    print("\n--- Explicit vs. Clean Content Summary ---")
    print(song_summary.groupby('is_explicit')[['days_present', 'span_days', 'days_to_peak', 'peak_pos', 'avg_popularity']].mean())

    print("\n--- Single vs. Album Track Summary ---")
    print(song_summary.groupby('album_type')[['days_present', 'span_days', 'days_to_peak', 'peak_pos', 'avg_popularity']].mean())

    print("\n================================================================")
    print("ANALYSIS COMPLETED SUCCESSFULLY")
    print("================================================================")

    return song_summary, df

if __name__ == '__main__':
    song_summary, df_cleaned = run_analysis()

STARTING SPAIN TOP 50 LIFECYCLE & CONTENT MATURITY ANALYSIS
-> Loaded 27800 rows. Cleaned duplicates: 50 rows removed.


C:\Users\nidhi\AppData\Local\Temp\ipykernel_15304\515217575.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  time_to_peak = df.groupby('song_id').apply(calc_time_to_peak)



----------------------------------------------------------------
KEY PERFORMANCE INDICATORS (KPIS)
----------------------------------------------------------------
• Average Days on Playlist:       48.26 Days
• Entry-to-Peak Time:             9.48 Days
• Playlist Churn Rate:            3.44% per day
• Retention Stability Index:      0.917 (91.7%)
• Explicit Content Lifecycle Score: 0.909 (45.7d explicit vs 50.3d clean)
• Single vs Album Longevity Ratio:  1.54x (58.7d single vs 38.2d album)
----------------------------------------------------------------

--- Explicit vs. Clean Content Summary ---
             days_present  span_days  days_to_peak   peak_pos  avg_popularity
is_explicit                                                                  
False           50.258462  59.876923          9.64  22.052308       71.131337
True            45.664000  54.280000          9.28  21.456000       69.260330

--- Single vs. Album Track Summary ---
            days_present  span_days  days_t